[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/feature/notebook-rebuild/notebooks/patterns/openai_agents_patterns.ipynb)

# Seven matched patterns with the OpenAI Agents SDK

`Agent`, `Runner`, typed outputs, tools and handoff declarations remain visible. A thin SDK `Model` adapter converts the shared deterministic fixtures into SDK responses, so no OpenAI key or network access is used. Mock runtime is under one minute on CPU; local execution is practical.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "openai-agents==0.17.8", "pydantic==2.12.5"],
        check=True,
    )

In [ ]:
import asyncio
import json
from pathlib import Path
from typing import ClassVar

from agents import Agent, Runner, function_tool, set_tracing_disabled
from agents.items import ModelResponse as SDKModelResponse
from agents.models.interface import Model as SDKModel
from agents.usage import InputTokensDetails, Usage
from openai.types.responses import ResponseOutputMessage, ResponseOutputText
from pydantic import BaseModel, ConfigDict, Field

set_tracing_disabled(True)
if InputTokensDetails.model_fields["cache_write_tokens"].is_required():
    InputTokensDetails.model_fields["cache_write_tokens"].default = 0
    InputTokensDetails.model_rebuild(force=True)
ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists() and "google.colab" in sys.modules:
    ROOT = Path("/content/agentic-ai-tutorial")
    if not ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "feature/notebook-rebuild",
                "https://github.com/jmdvinodjmd/agentic-ai-tutorial.git",
                str(ROOT),
            ],
            check=True,
        )
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run from the cloned repository.")
sys.path.insert(0, str(ROOT / "src"))
from agentic_tutorial.models import DeterministicMockClient  # noqa: E402
from agentic_tutorial.models.providers.fixtures import ScriptedScenarioFixture  # noqa: E402
from agentic_tutorial.schemas import (  # noqa: E402
    CriticDecision,
    Message,
    MessageRole,
    PlanDecision,
    RouteDecision,
    ToolDefinition,
)

fixture_data = json.loads(
    (ROOT / "data/research_assistant/pattern_mock_scenarios.json").read_text()
)
catalogue = fixture_data["catalogue"]

## Typed contracts, bounded tool and SDK adapter
The adapter only translates provider response shape. It does not route, loop or execute tools. A temporary compatibility default handles an upstream SDK/OpenAI type mismatch in the pinned July 2026 versions.

In [ ]:
class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


class Claim(StrictModel):
    schema_id: ClassVar[str] = "tutorial.claim.v1"
    claim: str
    source_id: str
    supported: bool


class Answer(StrictModel):
    schema_id: ClassVar[str] = "tutorial.pattern_answer.v1"
    answer: str
    source_ids: tuple[str, ...]


class ParallelDecision(StrictModel):
    schema_id: ClassVar[str] = "tutorial.parallel.v1"
    queries: tuple[str, ...] = Field(min_length=2)
    aggregation: str


class WorkerAssignment(StrictModel):
    worker: str
    query: str


class OrchestrationDecision(StrictModel):
    schema_id: ClassVar[str] = "tutorial.orchestration.v1"
    assignments: tuple[WorkerAssignment, ...] = Field(min_length=1, max_length=2)


OrchestrationDecision.model_rebuild(_types_namespace={"WorkerAssignment": WorkerAssignment})


def model_for(name):
    return DeterministicMockClient(
        ScriptedScenarioFixture.model_validate(
            {
                "fixture_version": fixture_data["fixture_version"],
                "scenario": name,
                "provenance": fixture_data["provenance"],
                "responses": fixture_data["scenarios"][name],
            }
        )
    )


def user(text):
    return Message(role=MessageRole.USER, content=text)


def search(query, limit=3):
    terms = set(query.casefold().split())
    return [r for r in catalogue if terms & set(" ".join(r["topics"]).casefold().split())][:limit]


@function_tool
def search_catalogue(query: str, max_results: int) -> str:
    """Search bounded evidence.

    Args:
        query: Search terms.
        max_results: Maximum records.
    """
    return json.dumps(search(query, max_results))


search_contract = ToolDefinition(
    name="search_catalogue",
    description="Search bounded evidence.",
    parameters={
        "type": "object",
        "properties": {"query": {"type": "string"}, "max_results": {"type": "integer"}},
        "required": ["query", "max_results"],
    },
)

In [ ]:
class FixtureSDKModel(SDKModel):
    def __init__(self, scenario):
        self.core = model_for(scenario)
        self.calls = 0

    async def get_response(
        self,
        system_instructions,
        input,
        model_settings,
        tools,
        output_schema,
        handoffs,
        tracing,
        *,
        previous_response_id,
        conversation_id,
        prompt,
    ):
        response = await self.core.generate([user(str(input))])
        self.calls += 1
        text = json.dumps(response.structured_output, sort_keys=True)
        item = ResponseOutputMessage(
            id=response.response_id,
            content=[
                ResponseOutputText(annotations=[], text=text, type="output_text", logprobs=[])
            ],
            role="assistant",
            status="completed",
            type="message",
        )
        return SDKModelResponse(output=[item], usage=Usage(), response_id=response.response_id)

    async def stream_response(self, *args, **kwargs):
        if False:
            yield None


async def run_typed(name, instructions, model, output_type, prompt):
    agent = Agent(name=name, instructions=instructions, model=model, output_type=output_type)
    result = await Runner.run(agent, prompt, max_turns=2)
    return result.final_output

## 1–4. Chaining, routing, parallel work and ReAct
Runner performs typed stages; deterministic Python validates the selected route and executes the function-tool contract. Unknown routes and tool names stop safely.

In [ ]:
chain_model = FixtureSDKModel("prompt_chaining")
claim = await run_typed(
    "Extractor", "Extract one supported claim and source ID.", chain_model, Claim, "Extract."
)
chain_answer = await run_typed(
    "Synthesiser", "Use only the supplied claim.", chain_model, Answer, str(claim.model_dump())
)
chain_state = {
    "trace": ["extract", "synthesise"],
    "stop": "criteria_met",
    "answer": chain_answer.model_dump(),
}
research_agent = Agent(
    name="Research specialist",
    instructions="Use only bounded catalogue evidence.",
    model=FixtureSDKModel("routing"),
)
clarify_agent = Agent(
    name="Clarifier", instructions="Request missing scope.", model=research_agent.model
)
route_model = research_agent.model
triage_agent = Agent(
    name="Triage",
    instructions="Return a typed route.",
    model=route_model,
    output_type=RouteDecision,
    handoffs=[research_agent, clarify_agent],
)
route_result = await Runner.run(triage_agent, "Route the evidence question.", max_turns=2)
route = route_result.final_output
route_records = search("meal planning portion") if route.route == "research" else []
route_state = {
    "results": route_records,
    "trace": ["route_validated", route.route],
    "stop": "criteria_met" if route.route in {"research", "clarify"} else "safe_fallback",
}
parallel_model = FixtureSDKModel("parallelisation")
parallel_decision = await run_typed(
    "Parallel planner",
    "Propose two independent queries.",
    parallel_model,
    ParallelDecision,
    "Plan searches.",
)
batches = await asyncio.gather(
    *(asyncio.to_thread(search, q, 2) for q in parallel_decision.queries)
)
parallel_state = {
    "source_ids": sorted({r["source_id"] for batch in batches for r in batch}),
    "trace": ["fan_out", "validated_union"],
    "stop": "criteria_met",
}
react_model = model_for("react")
react_response = await react_model.generate([user("Choose one tool.")], tools=[search_contract])
call = react_response.tool_calls[0]
react_state = {"steps": 0, "trace": ["invalid_tool"], "stop": "invalid_tool"}
if call.name == search_catalogue.name:
    react_state = {
        "steps": 1,
        "records": search(call.arguments["query"], call.arguments["max_results"]),
        "trace": ["tool_validated", "tool_executed"],
        "stop": "criteria_met",
    }

## 5–7. Planner–executor, critic–reviser and orchestrator–worker
Typed Runner calls propose plans, critiques and assignments. Revision is capped at one; named workers receive only read-only search. SDK handoff declarations document transfer topology, while validated Python controls whether transfer is authorised.

In [ ]:
plan = await run_typed(
    "Planner",
    "Create a three-step dependency plan.",
    FixtureSDKModel("planner_executor"),
    PlanDecision,
    "Plan.",
)
plan_valid = len(plan.steps) == 3 and all(
    plan.steps[i].depends_on == (plan.steps[i - 1].step_id,) for i in (1, 2)
)
plan_state = {
    "plan": plan.model_dump(),
    "trace": ["plan_validated", "executed"] if plan_valid else ["dependency_stop"],
    "stop": "criteria_met" if plan_valid else "invalid_plan",
}
critic_sdk_model = FixtureSDKModel("critic_reviser")
draft = await run_typed("Writer", "Draft from evidence.", critic_sdk_model, Answer, "Draft.")
critique = await run_typed(
    "Critic",
    "Check support and conditions.",
    critic_sdk_model,
    CriticDecision,
    str(draft.model_dump()),
)
revisions = 0
if not critique.accepted:
    draft = await run_typed(
        "Reviser", "Revise once.", critic_sdk_model, Answer, str(critique.issues)
    )
    revisions = 1
critic_state = {
    "answer": draft.model_dump(),
    "revisions": revisions,
    "trace": ["draft", "critic", "revision"],
    "stop": "criteria_met" if draft.source_ids else "revision_budget_exhausted",
}
orchestrator_sdk_model = FixtureSDKModel("orchestrator_worker")
assignments = await run_typed(
    "Orchestrator",
    "Assign named bounded workers.",
    orchestrator_sdk_model,
    OrchestrationDecision,
    "Assign.",
)
allowed = {"intervention_reviewer", "planning_reviewer"}
assignment_valid = all(a.worker in allowed for a in assignments.assignments)
worker_batches = (
    await asyncio.gather(*(asyncio.to_thread(search, a.query, 2) for a in assignments.assignments))
    if assignment_valid
    else []
)
worker_answer = (
    await run_typed(
        "Synthesis agent",
        "Combine worker evidence.",
        orchestrator_sdk_model,
        Answer,
        str(worker_batches),
    )
    if assignment_valid
    else None
)
orchestrator_state = {
    "worker_count": len(worker_batches),
    "answer": worker_answer.model_dump() if worker_answer else None,
    "trace": ["assign", "workers", "synthesise"] if assignment_valid else ["invalid_assignment"],
    "stop": "criteria_met" if assignment_valid else "invalid_assignment",
}

## Evaluation summary
All model outputs crossed typed SDK or shared tool-call boundaries and all loops stopped explicitly. Limitations: the fixture adapter tests orchestration deterministically, not hosted-model tool-call quality; SDK package compatibility is version-sensitive.

In [ ]:
pattern_evaluations = {
    "prompt_chaining": chain_state["stop"] == "criteria_met",
    "routing": len(route_state["results"]) == 2,
    "parallelisation": parallel_state["source_ids"] == ["food-waste-001", "food-waste-002"],
    "react": react_state["stop"] == "criteria_met" and react_state["steps"] <= 2,
    "planner_executor": plan_state["stop"] == "criteria_met",
    "critic_reviser": critic_state["revisions"] == 1,
    "orchestrator_worker": orchestrator_state["worker_count"] == 2,
}
pattern_limitations = {
    "prompt_chaining": "early errors propagate",
    "routing": "valid routes can be semantically wrong",
    "parallelisation": "branches can duplicate work",
    "react": "tool loops require strict budgets",
    "planner_executor": "invalid dependencies stop execution",
    "critic_reviser": "one revision may be insufficient",
    "orchestrator_worker": "worker findings can conflict",
}
assert set(pattern_limitations) == set(pattern_evaluations)
assert all(pattern_evaluations.values()), pattern_evaluations
{
    "framework": "OpenAI Agents SDK",
    "evaluation": pattern_evaluations,
    "limitations": pattern_limitations,
    "traces": {
        "chain": chain_state["trace"],
        "react": react_state["trace"],
        "orchestrator": orchestrator_state["trace"],
    },
}